In [12]:
from pathlib import Path
import sys
import pandas as pd
import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

sys.path.insert(0, str(Path('..').resolve())) # Add parent directory to path for imports

from agents import SingleShotAgent, MultiShotAgent, ZeroShotAgent, ImageEditAgent
from lib.render import render_image
from lib.types import Spec
from lib.ai import gemini_score_aesthetic

In [7]:
specs_dir = Path('../datasets/specs/') # Load canva specs directory
spec_paths = sorted(list(specs_dir.glob('*/spec.json'))) # Get all spec directories
specs_df = pd.DataFrame({
    'template_id': [p.parent.name for p in spec_paths],
    'spec_path': spec_paths
})

print(f"Found {len(specs_df)} design specs")
specs_df.head()

Found 100 design specs


,template_id,spec_path
0,1067w-IQprojiENUA,../datasets/specs/1067w-IQprojiENUA/spec.json
1,1131w-02GXIIidf_4,../datasets/specs/1131w-02GXIIidf_4/spec.json
2,1131w-120Kpwxx3eY,../datasets/specs/1131w-120Kpwxx3eY/spec.json
3,1131w-50t2MzJLmsM,../datasets/specs/1131w-50t2MzJLmsM/spec.json
4,1131w-9cN5biKALLQ,../datasets/specs/1131w-9cN5biKALLQ/spec.json


In [10]:
AGENT_MODEL = "gemini/gemini-2.5-pro"

specs_dir = Path('../datasets/specs')
edits_dir = Path('../edits')

def process_single_edit(row, agent_type):
    """Process a single edit: agent edit (handles assets & rendering) -> evaluate."""
    
    template_id = row['template_id']
    edit_type_id = row['edit_type_id']
    edit_type = row['edit_type']
    instruction = row['instruction']
    
    # Construct paths
    spec_path = specs_dir / template_id / 'spec.json'
    agent_id = agent_type.lower().replace("agent", "")
    edit_id = f"{template_id}_t{edit_type_id}_{agent_id}"
    edit_dir = edits_dir / edit_id
    output_path = edit_dir / 'spec.json'
    
    # Skip if already exists
    if output_path.exists():
        print(f"⊙ {edit_id} already exists, skipping")
        render_path = edit_dir / 'render.png'
        if render_path.exists():
            aesthetic_score = gemini_score_aesthetic(render_path)
            return {
                'design': template_id,
                'template_id': edit_type_id,
                'template_name': edit_type,
                'categories': row['categories'],
                'instruction': instruction,
                'agent': agent_id,
                'model': AGENT_MODEL,
                'edit_id': edit_id,
                'edit_path': str(output_path),
                'render_path': str(render_path),
                'aesthetic_score': aesthetic_score,
            }
        else:
            print(f"No render found for {edit_id} {template_id}, skipping")
            return None
    
    # Create agent based on type
    if agent_type == "ZeroShotAgent":
        agent = ZeroShotAgent(model=AGENT_MODEL, verbose=False)
    elif agent_type == "SingleShotAgent":
        agent = SingleShotAgent(model=AGENT_MODEL, verbose=False)
    elif agent_type == "MultiShotAgent":
        agent = MultiShotAgent(model=AGENT_MODEL, verbose=False)
    elif agent_type == "ImageEditAgent":
        agent = ImageEditAgent(model="nano-banana", verbose=False)
    else:
        raise ValueError(f"Unknown agent type: {agent_type}")
    
    # Agent handles EVERYTHING: copies assets, edits, renders
    # print(f"→ Processing {edit_id}...")
    agent.edit(spec_path=spec_path, instruction=instruction, output_path=output_path)
    
    # Evaluate aesthetic score
    render_path = edit_dir / 'render.png'
    aesthetic_score = gemini_score_aesthetic(render_path) if render_path.exists() else None
    
    print(f"✓ {edit_id} (score: {aesthetic_score})")
    
    return {
        'design': template_id,
        'template_id': edit_type_id,
        'template_name': edit_type,
        'categories': row['categories'],
        'instruction': instruction,
        'agent': agent_id,
        'model': AGENT_MODEL,
        'edit_path': str(output_path),
        'render_path': str(render_path),
        'aesthetic_score': aesthetic_score,
    }
    

## Test transparent layer editing

In [ ]:
visualiz

Testing on 5 designs
Instruction: 'make it more dream-like'
Output directory: ../datasets/edits

Test designs:
  - 1067w-IQprojiENUA
  - 1131w-02GXIIidf_4
  - 1131w-120Kpwxx3eY
  - 1131w-50t2MzJLmsM
  - 1131w-9cN5biKALLQ


In [ ]:
    # Process edits for each agent type
all_results = []
agent_type = "SingleShotAgent"
print(f"\n{'='*80}")
print(f"Processing with {agent_type}")
print(f"{'='*80}\n")

# Process edits (parallel with ThreadPoolExecutor)
results = []
max_workers = int(os.getenv("EDIT_MAX_WORKERS", "12"))
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(process_single_edit, row, agent_type) for _, row in df_sample.iterrows()]
    for fut in tqdm(as_completed(futures), total=len(futures), desc=f"{agent_type}", unit="edit"):
        try:
            result = fut.result()
            if result:
                results.append(result)
        except Exception as e:
            print(f"✗ Worker error: {e}")

all_results.extend(results)
print(f"\n✓ {agent_type}: Processed {len(results)} edits")

# Save combined results
df_results = pd.DataFrame(all_results)
results_csv = Path('edit_results.csv')
df_results.to_csv(results_csv, index=False)
print(f"\n{'='*80}")
print(f"✓ Total: Processed {len(df_results)} edits across {len(AGENT_TYPES)} agents")
print(f"✓ Saved to {results_csv.resolve()}")
print(f"{'='*80}\n")
print(df_results.head())